# BearingFault — Inference API Demo

End-to-end inference from a **raw `.mat` file** to a fault diagnosis result.

**Flow:**
```
new .mat file  (sensor recording)
      |
      v
load_mat_file()          — parse signal + metadata
      |
      v
extract_all_features()   — 171 DSP features (time / freq / WPD / envelope)
      |
      v
selector.transform()     — 171 -> ~85 features  (loaded from MLflow)
      |
      v
model.predict()          — Healthy / OR_damage / IR_damage  (loaded from MLflow)
```

**Step 1** — Load best model from MLflow  
**Step 2** — Direct Python inference on a real `.mat` file  
**Step 3** — Start FastAPI server  
**Step 4** — Call the API via HTTP POST  
**Step 5** — Simulate factory PLC/SCADA call

In [1]:
import importlib.util, os, sys, threading, time, requests
from pathlib import Path
import numpy as np
import mlflow

os.environ["MLFLOW_ENABLE_ARTIFACTS_PROGRESS_BAR"] = "false"   # suppress download progress bars

_BASE_DIR  = Path().resolve()
_UTILS     = _BASE_DIR / "utils"
MLRUNS_URI = f"file:///{_BASE_DIR / 'mlruns'}"
EXPERIMENT = "paderborn-bearing-fault"
CLASS_NAMES = ["Healthy", "OR_damage", "IR_damage"]
API_URL    = "http://127.0.0.1:8000"

# Load utils modules (same approach as BearingFault_Training.ipynb)
def _load_module(name, path):
    spec = importlib.util.spec_from_file_location(name, path)
    mod  = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)
    globals().update({k: v for k, v in vars(mod).items() if not k.startswith('_')})

_load_module("data_loader",   _UTILS / "data_loader.py")
_load_module("dsp_features",  _UTILS / "dsp_features.py")

print("Ready.")

Ready.


## Step 1 — Load Model Directly from MLflow

Verify the saved model loads correctly before starting the API server.

In [2]:
mlflow.set_tracking_uri(MLRUNS_URI)

_runs = mlflow.search_runs(
    experiment_names=[EXPERIMENT],
    filter_string="attributes.status = 'FINISHED'",
    order_by=["metrics.best_accuracy DESC"],   # 选准确率最高的 run
)

if _runs.empty:
    raise RuntimeError("No completed MLflow runs found. Run BearingFault_Training.ipynb first.")

_run_id   = _runs.iloc[0]["run_id"]
_accuracy = _runs.iloc[0].get("metrics.best_accuracy", "?")
_model    = _runs.iloc[0].get("params.best_model", "?")

print(f"Selected run : {_run_id[:8]}...  (highest accuracy among {len(_runs)} runs)")
print(f"Best model   : {_model}")
print(f"Accuracy     : {_accuracy:.4f}" if isinstance(_accuracy, float) else f"Accuracy: {_accuracy}")
print()
print("All completed runs:")
_cols = ["run_id", "params.best_model", "metrics.best_accuracy", "start_time"]
_available = [c for c in _cols if c in _runs.columns]
display(_runs[_available].head(5).style.format(
    {"metrics.best_accuracy": "{:.4f}", "run_id": lambda x: x[:8] + "..."}
))

sel   = mlflow.sklearn.load_model(f"runs:/{_run_id}/feature_selector")
model = mlflow.sklearn.load_model(f"runs:/{_run_id}/best_model")
print(f"\nModels loaded successfully.")

c:\Users\20215518\Anaconda3\envs\ds-py311\Lib\site-packages\mlflow\tracking\_tracking_service\utils.py:184: FutureWarning: The filesystem tracking backend (e.g., './mlruns') is deprecated as of February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://mlflow.org/docs/latest/self-hosting/migrate-from-file-store for migration guidance.
  return FileStore(store_uri, store_uri)


Selected run : 0d20d1c6...  (highest accuracy among 8 runs)
Best model   : XGB
Accuracy     : 0.9554

All completed runs:


,run_id,params.best_model,metrics.best_accuracy,start_time
0,0d20d1c6...,XGB,0.9554,2026-03-13 10:00:35.658000+00:00
1,e6dab83f...,XGB,0.9554,2026-03-12 21:27:15.032000+00:00
2,da75888f...,RF,0.9000,2026-03-12 14:55:46.844000+00:00
3,ca4396ca...,RF,0.9000,2026-03-12 14:28:14.627000+00:00
4,409b9466...,RF,0.9000,2026-03-12 13:50:42.168000+00:00



Models loaded successfully.


## Step 2 — Direct Inference (no API)

Test the model directly in Python first to establish a ground truth,
then compare against the API response in Step 4.

In [3]:
# --- Step 2: Direct inference from a real .mat file ---
# Change MAT_FILE to any .mat file in the dataset to test a different signal.
# Examples:
#   Healthy    : paderborn_data/mat/K001/N15_M07_F10_K001_1.mat
#   OR damage  : paderborn_data/mat/KA04/N15_M07_F10_KA04_1.mat
#   IR damage  : paderborn_data/mat/KI04/N15_M07_F10_KI04_1.mat

MAT_FILE = _BASE_DIR / "paderborn_data" / "mat" / "K001" / "N15_M07_F10_K001_1.mat"

if not MAT_FILE.exists():
    raise FileNotFoundError(f"File not found: {MAT_FILE}\n"
                            "Run BearingFault_Training.ipynb Section 1 first to download data.")

# Step 1: load raw signal
sig = load_mat_file(str(MAT_FILE))
print(f"Loaded  : {MAT_FILE.name}")
print(f"Bearing : {sig.bearing_code}  |  True label : {CLASS_NAMES[sig.label_3class]}")
print(f"Signal  : vibration {sig.vibration.shape}  current {sig.phase_current_1.shape}")

# Step 2: extract 171 DSP features
rpm          = OPERATING_CONDITIONS[sig.setting]['speed_rpm']
char_freqs   = calc_characteristic_frequencies(rpm)
feats        = extract_features_from_bearing(sig, use_current=True, use_vibration=True,
                                              characteristic_freqs=char_freqs)
X_single     = np.array([list(feats.values())], dtype=np.float32)   # (1, 171)
print(f"\nFeature vector : {X_single.shape}  (171 raw DSP features)")

# Step 3: selector + model
X_single_fs  = sel.transform(X_single)                               # (1, ~85)
y_direct     = model.predict(X_single_fs)
label_direct = CLASS_NAMES[y_direct[0]]

print(f"After selection: {X_single_fs.shape}")
print(f"\nPrediction : {label_direct}  (true: {CLASS_NAMES[sig.label_3class]})")
correct = label_direct == CLASS_NAMES[sig.label_3class]
print(f"Correct    : {'YES' if correct else 'NO'}")

# Keep X_sample for API comparison in Step 4
X_sample = X_single

Loaded  : N15_M07_F10_K001_1.mat
Bearing : K001  |  True label : Healthy
Signal  : vibration (256001,)  current (256001,)

Feature vector : (1, 183)  (171 raw DSP features)
After selection: (1, 92)

Prediction : Healthy  (true: Healthy)
Correct    : YES


## Step 3 — Start FastAPI Server

Launch `utils/inference_api.py` in a background thread.
The server stays alive for the rest of this notebook session.

In [4]:
import uvicorn

# Load the FastAPI app from utils/inference_api.py
spec = importlib.util.spec_from_file_location('inference_api', _UTILS / 'inference_api.py')
_api_mod = importlib.util.module_from_spec(spec)
spec.loader.exec_module(_api_mod)
app = _api_mod.app

# Start uvicorn in a daemon thread so it doesn't block the notebook
_config = uvicorn.Config(app, host='127.0.0.1', port=8000, log_level='warning')
_server = uvicorn.Server(_config)
_thread = threading.Thread(target=_server.run, daemon=True)
_thread.start()

# Wait for server to be ready
time.sleep(2)
print(f'API server running at {API_URL}')
print(f'Docs available at    {API_URL}/docs')

Model loaded from run: 0d20d1c6...
Best model: XGB
API server running at http://127.0.0.1:8000
Docs available at    http://127.0.0.1:8000/docs


## Step 4 — Call the API

Send the same 5 samples as HTTP POST requests and compare with direct inference results.

In [5]:
# Health check
_r = requests.get(f'{API_URL}/health')
print('Health check:', _r.json())

Health check: {'status': 'ok', 'run_id': '0d20d1c6'}


In [6]:
# Send the 5 samples as a batch POST request
_payload = {'features': X_sample.tolist()}
_r = requests.post(f'{API_URL}/predict', json=_payload)
_r.raise_for_status()
_result = _r.json()

print('API response:')
print(f"  run_id      : {_result['run_id']}")
print(f"  predictions : {_result['predictions']}")
print(f"  labels      : {_result['labels']}")

# Verify API matches direct inference
_api_preds = np.array(_result['predictions'])
assert np.array_equal(_api_preds, y_direct), 'API predictions differ from direct inference!'
print('\nAPI predictions match direct inference OK')

API response:
  run_id      : 0d20d1c6
  predictions : [0]
  labels      : ['Healthy']

API predictions match direct inference OK


## Step 5 — Interactive Test

Simulate how a factory PLC/SCADA system would call the API with a single new signal.

In [ ]:
# --- Step 5: Simulate factory PLC/SCADA call via /predict_mat ---
# The client only needs to supply the raw .mat file.
# DSP feature extraction (171 features) is done entirely server-side.
# Change NEW_MAT_FILE to any .mat file in the dataset to diagnose a different bearing.
NEW_MAT_FILE = _BASE_DIR / "paderborn_data" / "mat" / "KA04" / "N15_M07_F10_KA04_1.mat"

if not NEW_MAT_FILE.exists():
    raise FileNotFoundError(f"File not found: {NEW_MAT_FILE}")

# Load ground-truth label locally for verification only
sig_new    = load_mat_file(str(NEW_MAT_FILE))
true_label = CLASS_NAMES[sig_new.label_3class]

# POST the raw .mat file — no feature extraction needed on the client side
with open(NEW_MAT_FILE, "rb") as f:
    _r = requests.post(
        f"{API_URL}/predict_mat",
        files={"file": (NEW_MAT_FILE.name, f, "application/octet-stream")},
    )
_r.raise_for_status()
_out = _r.json()

print(f"Input file : {NEW_MAT_FILE.name}")
print(f"True label : {true_label}")
print(f"Prediction : {_out['labels'][0]}")
print(f"Correct    : {'YES' if _out['labels'][0] == true_label else 'NO'}")
print(f"Served by  : run {_out['run_id']}")

## Production Deployment

To deploy this API on a server (not inside a notebook):

```bash
# Option A — run directly with uvicorn
cd "c:\8. ds"
uvicorn utils.inference_api:app --host 0.0.0.0 --port 8000

# Option B — run via Docker (recommended)
docker compose up --build
```

**Call the API from any client** (factory PLC, Python script, etc.):

```bash
# Upload a raw .mat file — feature extraction is done server-side
curl -X POST http://<server-ip>:8000/predict_mat \
     -F "file=@N15_M07_F10_KA04_1.mat"
```

```python
# Python client — send raw .mat file, receive fault label
import requests
with open("N15_M07_F10_KA04_1.mat", "rb") as f:
    r = requests.post(
        "http://<server-ip>:8000/predict_mat",
        files={"file": (f.name, f, "application/octet-stream")},
    )
print(r.json())   # {"predictions": [1], "labels": ["OR_damage"], "run_id": "..."}
```

Interactive API docs (Swagger UI): `http://<server-ip>:8000/docs`